# Automation — the hands-on half

The practical companion to **`automation_slides.html`**. Two halves, and they answer different
questions:

* **Ray Serve** — how does a model get served with replicas, autoscaling and composition, without
  you writing a load balancer?
* **GitHub Actions** — what runs, automatically, when someone pushes?

| Part | Deck slides | What you do |
|---|---|---|
| 0 · Setup | 3 | install, sandbox |
| 1 · Ray Serve | 4–8 | a deployment, real HTTP, replicas, batching |
| 2 · Composition | 9 | two deployments calling each other |
| 3 · CI | 10–12 | a workflow that validates this repo, and really runs |
| 4 · Retrain & deploy | 13–16 | the pipeline you actually want on a schedule |

Everything runs in a throwaway `auto_demo/` folder; the last cell shuts Serve down.

## Step 0.1 · Install

In [1]:
!pip install -q 'ray[serve]' scikit-learn pandas requests

# Housekeeping, not part of the lesson: pandas prints "Pandas requires version ... of
# numexpr / bottleneck" when those are a version behind. That is about this machine, and
# joblib reprints it from every worker process, which buries the results.
import os, warnings
warnings.filterwarnings("ignore", message="Pandas requires version")
os.environ["PYTHONWARNINGS"] = "ignore:Pandas requires version"   # workers inherit this


## Step 0.2 · A sandbox to work in

In [2]:
import os, shutil, pathlib

BASE = pathlib.Path.cwd()          # the folder this notebook lives in
PROJ = BASE / "auto_demo"           # a throwaway sandbox, deleted by the last cell

if PROJ.exists():
    shutil.rmtree(PROJ)            # re-running this notebook is always safe
PROJ.mkdir(parents=True)
os.chdir(PROJ)
print("working inside:", os.getcwd())

working inside: /home/shamaseen/Desktop/Shai/qafza/free_Training/qafza-free-traning/11-automation/auto_demo


---
# Part 1 — Ray Serve   ·   deck slides 4–8

Session 8 showed actors: a class in its own process, holding state. **Ray Serve is a managed pool
of those actors with HTTP in front** — replicas, health checks, autoscaling and routing, without
you writing any of it.

In [3]:
import socket, time, numpy as np, pandas as pd, requests
import ray
from ray import serve

def free_port(start):
    for p in range(start, start + 200):
        with socket.socket() as s:
            if s.connect_ex(("127.0.0.1", p)) != 0:
                return p
    raise RuntimeError("no free port")

# Serve defaults to 8000. Pick a free port instead -- something else is often there
# already (an editor, another service), and the failure is confusing.
HTTP_PORT = free_port(8400)

ray.init(ignore_reinit_error=True, log_to_driver=False,
         include_dashboard=False, configure_logging=False)
serve.start(http_options={"host": "127.0.0.1", "port": HTTP_PORT})
SERVE_URL = f"http://127.0.0.1:{HTTP_PORT}"    # not BASE: the sandbox cell owns that name
print(f"serve listening on {SERVE_URL}")

/home/shamaseen/anaconda3/lib/python3.9/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(
INFO 2026-08-30 00:31:48,658 serve 1904978 -- Started Serve in namespace "serve".


serve listening on http://127.0.0.1:8400


Something worth serving first — the same churn model as everywhere else in this course.

In [4]:
# train something to serve
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import joblib

rng = np.random.default_rng(0); m = 3000
X = pd.DataFrame({"tenure_months": rng.integers(1, 72, m),
                  "monthly_charge": rng.normal(65, 20, m).round(2),
                  "support_calls": rng.poisson(1.2, m)})
s = (-0.04*X.tenure_months + 0.03*X.monthly_charge + 0.55*X.support_calls + rng.normal(0,.8,m))
y = (s > s.mean()).astype(int)
pipe = Pipeline([("sc", StandardScaler()),
                 ("rf", RandomForestClassifier(n_estimators=120, random_state=0))]).fit(X, y)
joblib.dump({"pipeline": pipe, "version": "3.0.0", "features": list(X.columns)}, "model.joblib")
print("model saved")

model saved


A Ray Serve **deployment** is an ordinary class. The decorator says how many copies
(**replicas**) to run; Ray handles the routing between them.

In [5]:
from fastapi import FastAPI
from pydantic import BaseModel, ConfigDict, Field
from typing import List, Literal

api = FastAPI(title="Churn API on Ray Serve")


class Customer(BaseModel):
    model_config = ConfigDict(extra="forbid")
    tenure_months:  int   = Field(ge=0, le=600)
    monthly_charge: float = Field(ge=0, le=10_000)
    support_calls:  int   = Field(ge=0, le=100)


@serve.deployment(
    num_replicas=2,                        # two processes, each with its own copy of the model
    ray_actor_options={"num_cpus": 1},
)
@serve.ingress(api)                        # Serve speaks FastAPI, so sessions 3 and 11 agree
class ChurnService:
    def __init__(self, artifact="model.joblib"):
        import joblib, os
        self.bundle = joblib.load(artifact)     # loaded ONCE per replica, at startup
        self.pid = os.getpid()

    @api.get("/health")
    def health(self):
        return {"status": "ok", "version": self.bundle["version"], "replica_pid": self.pid}

    @api.post("/predict")
    def predict(self, c: Customer):
        frame = pd.DataFrame([c.model_dump()])[self.bundle["features"]]
        p = float(self.bundle["pipeline"].predict_proba(frame)[0, 1])
        return {"churn": p >= .5, "probability": round(p, 4),
                "version": self.bundle["version"], "replica_pid": self.pid}


handle = serve.run(ChurnService.bind(), name="churn", route_prefix="/churn")
print("deployed")

INFO 2026-08-30 00:31:49,246 serve 1904978 -- Connecting to existing Serve app in namespace "serve". New http options will not be applied.
WARNING 2026-08-30 00:31:49,247 serve 1904978 -- The new client HTTP config differs from the existing one in the following fields: ['port', 'location']. The new HTTP config is ignored.
INFO 2026-08-30 00:31:50,355 serve 1904978 -- Application 'churn' is ready at http://127.0.0.1:8400/churn.


deployed


Wait for it to come up, then send it real HTTP.

In [6]:
import requests, time
for _ in range(60):
    try:
        r = requests.get(f"{SERVE_URL}/churn/health", timeout=2)
        if r.status_code == 200: break
    except Exception: time.sleep(1)
print("GET  /churn/health :", r.json())

r = requests.post(f"{SERVE_URL}/churn/predict",
                  json={"tenure_months": 2, "monthly_charge": 140.0, "support_calls": 9})
print("POST /churn/predict:", r.status_code, r.json())
assert r.status_code == 200

GET  /churn/health : {'status': 'ok', 'version': '3.0.0', 'replica_pid': 1905411}
POST /churn/predict: 200 {'churn': True, 'probability': 1.0, 'version': '3.0.0', 'replica_pid': 1905402}


Sixty requests, and a count of which process answered each one. Two process ids means two
replicas really are sharing the load.

In [7]:
# prove there really are two replicas, by watching which process answers
pids = {}
for _ in range(60):
    j = requests.post(f"{SERVE_URL}/churn/predict",
                      json={"tenure_months": 12, "monthly_charge": 60.0, "support_calls": 1}).json()
    pids[j["replica_pid"]] = pids.get(j["replica_pid"], 0) + 1
print("requests answered per replica pid:", pids)
assert len(pids) >= 2, "num_replicas=2 should mean two processes answering"
print("\nTwo processes, each with its own loaded model. Serve routed between them.")

requests answered per replica pid: {1905411: 33, 1905402: 27}

Two processes, each with its own loaded model. Serve routed between them.


Validation has not changed: it is the same Pydantic contract from session 3, so a bad request is
still refused with 422 rather than scored.

In [8]:
# validation still comes from Pydantic -- the same contract as session 3
bad = requests.post(f"{SERVE_URL}/churn/predict",
                    json={"tenure_months": -4, "monthly_charge": 60.0, "support_calls": 1})
print("bad request ->", bad.status_code)
print(bad.json()["detail"][0]["msg"] if bad.status_code == 422 else bad.text[:120])
assert bad.status_code == 422

bad request -> 422
Input should be greater than or equal to 0


## Step 1.1 · Autoscaling and batching, declaratively

Two of the things you would otherwise build by hand are configuration here.

In [9]:
from ray.serve import deployment

@serve.deployment(
    autoscaling_config={
        "min_replicas": 1,
        "max_replicas": 4,
        "target_ongoing_requests": 5,     # scale out when replicas are busier than this
    },
    max_ongoing_requests=10,
)
class Autoscaled:
    def __call__(self, request):
        return {"ok": True}

print("autoscaling_config accepted:")
print("  min 1, max 4 replicas, target 5 ongoing requests per replica")
print("\nServe adds and removes replicas itself. You declare the target, not the mechanism.")

autoscaling_config accepted:
  min 1, max 4 replicas, target 5 ongoing requests per replica

Serve adds and removes replicas itself. You declare the target, not the mechanism.


**Dynamic batching** collects requests arriving at the same moment and runs them through the model
in one call. It is usually the single biggest serving win, and it is a decorator.

In [10]:
# dynamic batching: the single most effective serving optimisation
@serve.deployment(num_replicas=1)
class BatchedScorer:
    def __init__(self, artifact="model.joblib"):
        import joblib
        self.bundle = joblib.load(artifact)

    @serve.batch(max_batch_size=32, batch_wait_timeout_s=0.02)
    async def score_many(self, rows: list):
        """Serve collects concurrent calls into ONE list, so the model runs once."""
        frame = pd.DataFrame(rows)[self.bundle["features"]]
        probs = self.bundle["pipeline"].predict_proba(frame)[:, 1]
        return [float(p) for p in probs]      # one result per input, same order

    async def __call__(self, request):
        body = await request.json()
        return {"probability": round(await self.score_many(body), 4)}


serve.run(BatchedScorer.bind(), name="batched", route_prefix="/batched")

import concurrent.futures, time
payload = {"tenure_months": 6, "monthly_charge": 90.0, "support_calls": 4}
t0 = time.perf_counter()
with concurrent.futures.ThreadPoolExecutor(max_workers=32) as ex:
    out = list(ex.map(lambda _: requests.post(f"{SERVE_URL}/batched", json=payload).json(),
                      range(64)))
dt = time.perf_counter() - t0
print(f"64 concurrent requests through a batching deployment: {dt*1000:.0f} ms total")
print(f"  {dt/64*1000:.2f} ms each; all agree: {len({o['probability'] for o in out}) == 1}")

INFO 2026-08-30 00:31:50,988 serve 1904978 -- Connecting to existing Serve app in namespace "serve". New http options will not be applied.
WARNING 2026-08-30 00:31:50,988 serve 1904978 -- The new client HTTP config differs from the existing one in the following fields: ['port', 'location']. The new HTTP config is ignored.
INFO 2026-08-30 00:31:52,094 serve 1904978 -- Application 'batched' is ready at http://127.0.0.1:8400/batched.


64 concurrent requests through a batching deployment: 526 ms total
  8.22 ms each; all agree: True


---
# Part 2 — Composition   ·   deck slides 9

Real systems are not one model. A preprocessor, two models and a rule that combines them —
each with its own scaling needs. Serve lets one deployment hold a **handle** to another.

In [11]:
@serve.deployment(num_replicas=1)
class RiskModel:
    def __call__(self, features: dict) -> float:
        return min(1.0, features["support_calls"] / 10)


@serve.deployment(num_replicas=1)
class ValueModel:
    def __call__(self, features: dict) -> float:
        return min(1.0, features["monthly_charge"] / 200)


# NOTE: no @serve.ingress here. A class passed to ingress may not define __call__ --
# ingress is for FastAPI-routed classes, __call__ is for raw request handling.
@serve.deployment(num_replicas=1)
class Router:
    def __init__(self, risk, value):
        self.risk, self.value = risk, value     # handles to other deployments

    async def __call__(self, request):
        f = await request.json()
        r = await self.risk.remote(f)           # each call may land on a different node
        v = await self.value.remote(f)
        return {"risk": round(r, 3), "value": round(v, 3),
                "action": "call now" if r > .5 and v > .4 else "monitor"}


graph = Router.bind(RiskModel.bind(), ValueModel.bind())
serve.run(graph, name="composed", route_prefix="/composed")

r = requests.post(f"{SERVE_URL}/composed",
                  json={"support_calls": 8, "monthly_charge": 150.0})
print("POST /composed ->", r.status_code, r.json())
assert r.status_code == 200 and "action" in r.json()
print("\nThree deployments, one endpoint. Each can scale independently.")

INFO 2026-08-30 00:31:52,629 serve 1904978 -- Connecting to existing Serve app in namespace "serve". New http options will not be applied.
WARNING 2026-08-30 00:31:52,629 serve 1904978 -- The new client HTTP config differs from the existing one in the following fields: ['port', 'location']. The new HTTP config is ignored.
INFO 2026-08-30 00:31:53,736 serve 1904978 -- Application 'composed' is ready at http://127.0.0.1:8400/composed.


POST /composed -> 200 {'risk': 0.8, 'value': 0.75, 'action': 'call now'}

Three deployments, one endpoint. Each can scale independently.


What is deployed right now, straight from Serve's own status.

In [12]:
print("what is deployed right now:")
for name, info in serve.status().applications.items():
    print(f"  {name:10} {info.status}")

what is deployed right now:
  churn      RUNNING
  batched    RUNNING
  composed   RUNNING


---
# Part 3 — CI that actually guards something   ·   deck slides 10–12

A workflow is a YAML file in `.github/workflows/`. GitHub runs it on the events you name.

The most useful first workflow is not a deployment — it is the one that **stops broken work
reaching main**. For this course, that means: do the decks still parse and stay offline, and do
the notebooks still contain what they claim?

In [13]:
%%writefile ci-example.yml
name: validate

on:
  push:
    branches: [main]
  pull_request:
  workflow_dispatch:          # so you can also run it by hand

jobs:
  decks-and-notebooks:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"

      - name: every deck must be balanced and offline
        run: |
          python - <<'PY'
          import glob, re, sys
          bad = []
          for f in glob.glob("*/*_slides.html"):
              t = open(f).read()
              if t.count("<section") != t.count("</section>"):
                  bad.append(f"{f}: unbalanced sections")
              ext = re.findall(r'(?:src|href)="(?!#|data:)[^"]+"', t)
              if ext:
                  bad.append(f"{f}: external reference {ext[:2]}")
              print(f"{f}: {t.count('<section')} slides")
          if bad:
              print("FAILURES:", *bad, sep="\n  "); sys.exit(1)
          PY

      - name: every notebook must be valid and carry its outputs
        run: |
          python - <<'PY'
          import glob, json, sys
          bad = []
          for f in glob.glob("*/*_tutorial.ipynb"):
              nb = json.load(open(f))
              code = [c for c in nb["cells"] if c["cell_type"] == "code"]
              errs = [o for c in nb["cells"] for o in c.get("outputs", [])
                      if o.get("output_type") == "error"]
              if errs:
                  bad.append(f"{f}: {len(errs)} stored error outputs")
              print(f"{f}: {len(nb['cells'])} cells, {len(code)} code, {len(errs)} errors")
          if bad:
              print("FAILURES:", *bad, sep="\n  "); sys.exit(1)
          PY

      - name: the course index must link every built session
        run: |
          python - <<'PY'
          import glob, os, re, sys
          readme = open("README.md").read()
          missing = [d for d in sorted(glob.glob("*/"))
                     if glob.glob(os.path.join(d, "*_slides.html"))
                     and f"({d.rstrip('/')})" not in readme]
          print("unlinked sessions:", missing or "none")
          sys.exit(1 if missing else 0)
          PY

Writing ci-example.yml


### Reading that workflow

| Line | Means |
|---|---|
| `on: push / pull_request` | when it runs. `pull_request` is the one that protects `main` |
| `workflow_dispatch` | adds a "Run workflow" button |
| `runs-on: ubuntu-latest` | a fresh VM, every time — nothing from your laptop |
| `uses: actions/checkout@v4` | pinned action version; `@v4` not `@main` |
| `run:` | shell in the runner |
| `sys.exit(1)` | **non-zero fails the job.** A check that cannot fail is decoration |

> **The habit that matters:** the check has to be able to fail. Plenty of CI jobs print warnings
> and exit 0, and nobody notices they stopped working.

This file is committed to the repository as `.github/workflows/validate.yml`, and it runs on
every push. The deck reports what GitHub actually said about it.

---
# Part 4 — Retraining and deployment   ·   deck slides 13–16

Now the workflow people actually ask for. Note how much of it is *guard rails* rather than
training.

In [14]:
%%writefile retrain-example.yml
name: retrain-and-deploy

on:
  schedule:
    - cron: "0 3 * * 1"        # 03:00 UTC every Monday
  workflow_dispatch:
    inputs:
      reason:
        description: why are you retraining by hand?
        required: true

concurrency:
  group: retrain              # never two retrains at once
  cancel-in-progress: false

jobs:
  retrain:
    runs-on: ubuntu-latest
    permissions:
      contents: read
      id-token: write          # OIDC: no long-lived cloud keys in secrets
    steps:
      - uses: actions/checkout@v4

      - uses: actions/setup-python@v5
        with: {python-version: "3.11", cache: pip}

      - run: pip install -r requirements.txt

      # session 6: the data is versioned, so this run is reproducible
      - name: fetch the dataset
        run: dvc pull
        env:
          AWS_ACCESS_KEY_ID: ${{ secrets.AWS_ACCESS_KEY_ID }}
          AWS_SECRET_ACCESS_KEY: ${{ secrets.AWS_SECRET_ACCESS_KEY }}

      # session 1: a leak-proof pipeline. session 7: the run is recorded
      - name: train
        run: python src/train.py
        env:
          MLFLOW_TRACKING_URI: ${{ vars.MLFLOW_TRACKING_URI }}

      # THE GATE. Without this the schedule will eventually ship a worse model.
      - name: refuse to promote a worse model
        run: |
          python - <<'PY'
          import json, sys
          new = json.load(open("metrics.json"))
          base = json.load(open("baseline.json"))
          delta = new["roc_auc"] - base["roc_auc"]
          print(f"new {new['roc_auc']:.4f}  baseline {base['roc_auc']:.4f}  delta {delta:+.4f}")
          if delta < -0.005:
              print("REFUSING to promote: the new model is worse."); sys.exit(1)
          PY

      - name: build and push the image        # session 4
        run: |
          TAG=ghcr.io/${{ github.repository }}:${{ github.sha }}
          docker build -t $TAG .
          echo ${{ secrets.GITHUB_TOKEN }} | docker login ghcr.io -u ${{ github.actor }} --password-stdin
          docker push $TAG

  deploy:
    needs: retrain
    runs-on: ubuntu-latest
    environment: production     # a required reviewer can be attached to this
    steps:
      - name: roll out
        run: echo "kubectl set image / serve deploy / terraform apply"

Writing retrain-example.yml


### What makes that workflow safe rather than merely automatic

| Piece | Without it |
|---|---|
| `concurrency: group` | two retrains race and one overwrites the other's model |
| the **metric gate** | a scheduled job eventually promotes a worse model, silently |
| `environment: production` | no human ever sees the change before customers do |
| `id-token: write` (OIDC) | long-lived cloud keys sitting in repository secrets |
| `dvc pull` at a pinned commit | "which data trained this?" has no answer |
| image tagged with `github.sha` | you cannot roll back to a specific commit |

> **The single most important line is the gate.** Automation without a quality gate does not
> remove human error, it removes the human who would have caught it.

## Step 4.1 · Where Serve and Actions meet

```
    push / schedule
          |
   GitHub Actions  --- train --> gate --> build image --> push to registry
          |
          v
    serve deploy / kubectl rollout      <-- Ray Serve picks up the new version
          |
          v
   session 10's Prometheus notices the new model_info version, and watches the
   prediction distribution for the shift that no test would have caught
```

Each session in this course is one box in that diagram. That is what session 13 assembles.

---
# Reference — worth knowing, not demonstrated here

| Topic | One-line version |
|---|---|
| Serve on Kubernetes | KubeRay, and `serve build` to produce a config |
| Canary / A-B between versions | two deployments, weighted routing in the router |
| Actions matrix builds | one job across several Python versions or OSes |
| Caching in CI | `actions/cache`, or `cache: pip` on setup-python |
| Self-hosted runners | when you need a GPU or private network access |
| Reusable workflows | `uses: ./.github/workflows/x.yml` — DRY across repos |
| Secrets vs variables | `secrets.*` is masked in logs; `vars.*` is not |
| OIDC to cloud | short-lived credentials instead of stored keys |

## Recap

| You wanted to… | Do this |
|---|---|
| serve a model with replicas | `@serve.deployment(num_replicas=N)` + `serve.run` |
| keep FastAPI validation | `@serve.ingress(app)` |
| scale on load | `autoscaling_config={...}` |
| make batching automatic | `@serve.batch(max_batch_size=..., batch_wait_timeout_s=...)` |
| compose several models | pass handles into a router deployment |
| stop broken work reaching main | a `pull_request` workflow that can **fail** |
| retrain on a schedule | `on: schedule` + `concurrency` + **a metric gate** |
| deploy with a human in the loop | `environment: production` with a reviewer |
| avoid stored cloud keys | `permissions: id-token: write` and OIDC |

## What to do at work tomorrow

1. Write the CI job that would have caught your last broken merge. One job, and make sure it can
   fail.
2. Add `concurrency` to anything that writes shared state.
3. If you retrain on a schedule, **add the gate today**. It is ten lines and it is the difference
   between automation and an accident waiting.
4. Put the git SHA in the image tag, so a rollback is a tag and not an archaeology project.

## Cleanup

In [15]:
import shutil, os, time

# delete the applications first, then the controller. Shutting the controller down
# while applications are live makes the raylet log a graceful_shutdown retry.
for app in ("churn", "batched", "composed"):
    try:
        serve.delete(app)
    except Exception as e:
        print(f"  {app}: {type(e).__name__}")
time.sleep(2)
serve.shutdown()
ray.shutdown()
print("serve and ray shut down")

os.chdir(BASE)
shutil.rmtree(PROJ, ignore_errors=True)
print("sandbox removed")


INFO 2026-08-30 00:31:53,778 serve 1904978 -- Deleting app ['churn']
INFO 2026-08-30 00:31:56,786 serve 1904978 -- Deleting app ['batched']
INFO 2026-08-30 00:31:59,794 serve 1904978 -- Deleting app ['composed']


(raylet) Task ServeController.graceful_shutdown failed. There are infinite retries remaining, so the task will be retried. Error: The actor is dead because it was killed by `ray.kill`.
serve and ray shut down
sandbox removed
